# **Key Performance Indicator (KPI) & Business Analysis**

##**Objective**

The objective of this notebook is to analyze the prepared road accident dataset using business-driven questions. The analysis focuses on identifying accident patterns, evaluating key performance indicators (KPIs), and generating actionable insights that can support road safety planning and decision-making.




# **Import Libraries**



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# **Dataset upload**

In [ ]:
df = pd.read_csv('/content/cleaned_road_accident.csv')
df.head()

,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score,severity_index,risk_quartile
0,0,Pune,Maharashtra,18.680826,73.93039,2023-10-22,5:00,5,Sunday,Weekend,...,high,weather,fatal,2,2,Non Peak Hour,No Festival,0.85,4,Q4
1,1,Mumbai,Maharashtra,18.817732,72.79085,2023-05-21,4:00,4,Sunday,Weekend,...,low,weather,major,4,3,Non Peak Hour,No Festival,0.10,12,Q1
2,2,Mumbai,Maharashtra,19.096890,72.81943,2024-07-10,13:00,13,Wednesday,Weekday,...,medium,weather,minor,1,1,Non Peak Hour,No Festival,0.45,1,Q2
3,3,Chandigarh,Punjab,30.787806,76.84750,2025-03-30,11:00,11,Sunday,Weekend,...,high,distraction,minor,5,2,Non Peak Hour,No Festival,0.65,10,Q4
4,4,Chennai,Tamil Nadu,12.965155,80.28331,2024-01-25,16:00,16,Thursday,Weekday,...,low,distraction,minor,2,1,Non Peak Hour,No Festival,0.10,2,Q1


# **Key Performance Indicator (KPI) Analysis**

# **Executive KPIs**

In [ ]:
total_accidents = len(df)

fatal_accidents = (
    df['accident_severity']
      .eq('fatal')
      .sum()
)

fatality_rate = (
    fatal_accidents
    / total_accidents
    *100
).round(2)

average_risk = df['risk_score'].mean().round(2)

average_severity = df['severity_index'].mean().round(2)

average_casualties = df['casualties'].mean().round(2)

Average_Vehicles_Involved = df['vehicles_involved'].mean().round()

print(f"Total accidents = {total_accidents}")
print(f"Fatal accidents = {fatal_accidents}")
print(f"fatality rate = {fatality_rate}")
print(f"average risk  = {average_risk }")
print(f"Average Vehicles Involved = {Average_Vehicles_Involved}")

Total accidents = 20000
Fatal accidents = 2987
fatality rate = 14.94
average risk  = 0.44
Average Vehicles Involved = 3.0


# **Severity Analysis**

 **Q1. Severity wise KPI Analysis**

In [ ]:
data = df.groupby('accident_severity').agg(
    total_accident = ('accident_severity','count'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({
          'avg_casualties': 2,
          'avg_risk_score': 2
      })

pd.DataFrame(data).reset_index()

,accident_severity,total_accident,avg_casualties,avg_risk_score
0,fatal,2987,3.00,0.61
1,major,5988,1.49,0.41
2,minor,11025,1.51,0.41


**Q2. Accident severity across states**

In [ ]:
result = (df.groupby(['state','accident_severity'])
          .agg(total_accident = ('accident_severity','count'),
               avg_casualties = ('casualties','mean')).round({'avg_casualties':2}))
result.reset_index().sort_values(by=['state', 'total_accident'],ascending=[True,False])

,state,accident_severity,total_accident,avg_casualties
2,Delhi,minor,1360,1.47
1,Delhi,major,691,1.48
0,Delhi,fatal,382,3.00
5,Karnataka,minor,1347,1.49
4,Karnataka,major,707,1.44
3,Karnataka,fatal,384,2.89
8,Maharashtra,minor,2725,1.51
7,Maharashtra,major,1527,1.55
6,Maharashtra,fatal,757,2.97
11,Punjab,minor,1419,1.51


**Q3. Traffic density × severity**

In [ ]:
result = (df.groupby(['traffic_density','accident_severity']).agg(
    total_accident = ('accident_severity','count'),
    avg_casualties = ('casualties','mean')).round({'avg_casualties':2}).reset_index()
    .sort_values(by=['traffic_density', 'total_accident'],ascending=[True,False]))
result

,traffic_density,accident_severity,total_accident,avg_casualties
2,high,minor,3912,1.51
1,high,major,2077,1.51
0,high,fatal,1045,2.99
5,low,minor,3890,1.48
4,low,major,2116,1.53
3,low,fatal,1061,3.00
8,medium,minor,3223,1.53
7,medium,major,1795,1.44
6,medium,fatal,881,3.02


**Q4. Weather vs severity**

In [ ]:
pd.pivot_table(
    data = df,
    index= 'weather',
    columns= 'accident_severity',
    values= 'accident_id',
    fill_value= 0,
    aggfunc= 'count'

).reset_index()

accident_severity,weather,fatal,major,minor
0,clear,992,2017,3681
1,fog,995,1975,3663
2,rain,1000,1996,3681


**Q5. Peak vs Non-Peak severity**

In [ ]:
pd.pivot_table(
    data = df,
    index = 'is_peak_hour',
    columns = 'accident_severity',
    values = 'accident_id',
    aggfunc = 'count',
    fill_value = 0
).reset_index()

accident_severity,is_peak_hour,fatal,major,minor
0,Non Peak Hour,2247,4483,8322
1,Peak Hour,740,1505,2703


**Q6. Weekend vs Weekday severity**

In [ ]:
pd.pivot_table(
    data = df,
    index = 'is_weekend',
    columns = 'accident_severity',
    values = 'accident_id',
    aggfunc = 'count',
    fill_value = 0
).reset_index()

accident_severity,is_weekend,fatal,major,minor
0,Weekday,2124,4304,7849
1,Weekend,863,1684,3176


**Q7. Traffic density vs severity**

In [ ]:
pd.crosstab(
    index=df['traffic_density'],
    columns = df['accident_severity']
).reset_index()

accident_severity,traffic_density,fatal,major,minor
0,high,1045,2077,3912
1,low,1061,2116,3890
2,medium,881,1795,3223


**Q8. Vehicles involved vs severity**

In [ ]:
df1 = df.groupby('vehicles_involved').agg(
    total_accidents = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby('vehicles_involved')
      .agg(fatal_accident = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on='vehicles_involved')
result['fatal_rate'] = (
    result['fatal_accident'] / result['total_accidents'] * 100
).round(2)

result = result.sort_values(
    by='fatal_rate',
    ascending=False
)

result

,vehicles_involved,total_accidents,total_casualties,fatal_accident,fatal_rate
3,4,3936,9103,609,15.47
1,2,4030,4679,604,14.99
4,5,3997,11572,597,14.94
0,1,4030,2309,597,14.81
2,3,4007,6866,580,14.47


**Q9. Traffic density & peak hour**

In [ ]:
df1 = df.groupby(['traffic_density','is_peak_hour']).agg(
    total_accidents = ('accident_severity','count'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2}).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby(['traffic_density','is_peak_hour'])
      .agg(fatal_accidents = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on=['traffic_density','is_peak_hour'])
result = result[['traffic_density','is_peak_hour','total_accidents','fatal_accidents','avg_casualties','avg_risk_score']]
result.sort_values(by=['traffic_density','fatal_accidents'],ascending=[True,False])

,traffic_density,is_peak_hour,total_accidents,fatal_accidents,avg_casualties,avg_risk_score
0,high,Non Peak Hour,4043,604,1.74,0.53
1,high,Peak Hour,2991,441,1.72,0.68
2,low,Non Peak Hour,6602,984,1.72,0.33
3,low,Peak Hour,465,77,1.74,0.49
4,medium,Non Peak Hour,4407,659,1.71,0.33
5,medium,Peak Hour,1492,222,1.76,0.48


**Q10. Severity across traffic density**

In [ ]:
result = df.groupby(['traffic_density','accident_severity']).agg(
    total_accidents = ('accident_severity','count'),
    Average_vehicles_involved = ('vehicles_involved','mean'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2,'Average_vehicles_involved': 2}).reset_index()

result = result[['traffic_density','accident_severity','total_accidents','Average_vehicles_involved','avg_casualties','avg_risk_score']]
result.sort_values(by=['traffic_density','total_accidents'],ascending=[True,False])

,traffic_density,accident_severity,total_accidents,Average_vehicles_involved,avg_casualties,avg_risk_score
2,high,minor,3912,3.00,1.51,0.57
1,high,major,2077,2.99,1.51,0.57
0,high,fatal,1045,2.99,2.99,0.76
5,low,minor,3890,2.97,1.48,0.31
4,low,major,2116,2.96,1.53,0.31
3,low,fatal,1061,3.00,3.00,0.51
8,medium,minor,3223,3.04,1.53,0.34
7,medium,major,1795,2.96,1.44,0.34
6,medium,fatal,881,3.02,3.02,0.54


# **State Analysis**

**Q1. Most dangerous states**

In [ ]:
df1 = df.groupby('state').agg(
    total_accident = ('accident_severity','count'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_risk_score':2}).reset_index()

fatal_df = df[df['accident_severity'] == 'fatal']
df2 = fatal_df.groupby('state').agg(
    fatal_accident = ('accident_severity','count')
).reset_index()
d = pd.merge(df1,df2,on='state')
d = d[['state','total_accident','fatal_accident','avg_risk_score']]
d.sort_values(by='fatal_accident', ascending=False)


,state,total_accident,fatal_accident,avg_risk_score
2,Maharashtra,5009,757,0.44
1,Karnataka,2438,384,0.44
0,Delhi,2433,382,0.44
3,Punjab,2577,378,0.44
6,West Bengal,2559,377,0.43
4,Tamil Nadu,2575,371,0.44
5,Telangana,2409,338,0.44


**Q2. State accident summary**

In [ ]:
pd.pivot_table(
    data = df,
    index = 'state',
    columns= 'accident_severity',
    values= 'accident_id',
    aggfunc= 'count',
    fill_value=0
).reset_index()

accident_severity,state,fatal,major,minor
0,Delhi,382,691,1360
1,Karnataka,384,707,1347
2,Maharashtra,757,1527,2725
3,Punjab,378,780,1419
4,Tamil Nadu,371,794,1410
5,Telangana,338,752,1319
6,West Bengal,377,737,1445


**Q3. Fatal accident rate**

In [ ]:
df1 = df.groupby('state').agg(
    total_accident = ('accident_severity','count'),
).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby('state')
      .agg(fatal_accident = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2,on='state')
result['fatal_rate'] = (
    result['fatal_accident'] / result['total_accident'] * 100
).round(2)
result = result.sort_values(
    by='fatal_rate',
    ascending=False
)

result = result[['state','total_accident','fatal_accident','fatal_rate']]
result

,state,total_accident,fatal_accident,fatal_rate
1,Karnataka,2438,384,15.75
0,Delhi,2433,382,15.70
2,Maharashtra,5009,757,15.11
6,West Bengal,2559,377,14.73
3,Punjab,2577,378,14.67
4,Tamil Nadu,2575,371,14.41
5,Telangana,2409,338,14.03


**Q4. Dangerous weekday in each state**

In [ ]:
df.groupby(['state','day_of_week']).agg(
    total_accident = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')).round({
    'avg_risk_score': 2,
}).reset_index().sort_values(by=['state','total_accident'],ascending=[True,False])

,state,day_of_week,total_accident,total_casualties,avg_risk_score
2,Delhi,Saturday,370,652,0.45
1,Delhi,Monday,360,584,0.42
3,Delhi,Sunday,360,633,0.45
4,Delhi,Thursday,352,602,0.45
6,Delhi,Wednesday,340,585,0.44
5,Delhi,Tuesday,337,600,0.44
0,Delhi,Friday,314,519,0.42
9,Karnataka,Saturday,374,616,0.43
12,Karnataka,Tuesday,370,601,0.46
13,Karnataka,Wednesday,356,626,0.43


**Q5.Dangerous states during peak traffic hours**

In [ ]:
df1 = df.groupby(['state','is_peak_hour']).agg(
    total_accidents = ('accident_severity','count'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_risk_score':2}).reset_index()

df2 = (df[df['accident_severity'] == 'fatal'].
       groupby(['state','is_peak_hour']).agg(
           fatal_accidents = ('accident_severity','count')
       ).reset_index()
       )

result = (pd.merge(df1,df2,on=['state','is_peak_hour']).
sort_values(by=['state','fatal_accidents'],ascending=[True,False]))

result = result[['state','is_peak_hour','total_accidents','fatal_accidents','avg_risk_score']]
result

,state,is_peak_hour,total_accidents,fatal_accidents,avg_risk_score
0,Delhi,Non Peak Hour,1844,281,0.38
1,Delhi,Peak Hour,589,101,0.61
2,Karnataka,Non Peak Hour,1817,279,0.38
3,Karnataka,Peak Hour,621,105,0.61
4,Maharashtra,Non Peak Hour,3811,569,0.38
5,Maharashtra,Peak Hour,1198,188,0.60
6,Punjab,Non Peak Hour,1923,286,0.39
7,Punjab,Peak Hour,654,92,0.60
8,Tamil Nadu,Non Peak Hour,1917,280,0.38
9,Tamil Nadu,Peak Hour,658,91,0.60


**Q6. Weekend accident risk**

In [ ]:
df1 = df.groupby(['state','is_weekend']).agg(
    total_accidents = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_risk_score':2}).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby(['state','is_weekend'])
      .agg(fatal_accidents = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on=['state','is_weekend'])
result = result[['state','is_weekend','total_accidents','total_casualties','fatal_accidents','avg_risk_score']]
result.sort_values(by=['state','fatal_accidents'],ascending=[True,False])

,state,is_weekend,total_accidents,total_casualties,fatal_accidents,avg_risk_score
0,Delhi,Weekday,1703,2890,264,0.43
1,Delhi,Weekend,730,1285,118,0.45
2,Karnataka,Weekday,1734,2944,270,0.44
3,Karnataka,Weekend,704,1184,114,0.43
4,Maharashtra,Weekday,3575,6198,550,0.44
5,Maharashtra,Weekend,1434,2518,207,0.43
6,Punjab,Weekday,1873,3195,277,0.44
7,Punjab,Weekend,704,1182,101,0.44
8,Tamil Nadu,Weekday,1838,3239,258,0.44
9,Tamil Nadu,Weekend,737,1328,113,0.44


**Q7. Average vehicles involved**

In [ ]:
result = (df.groupby('state').agg(
    total_accidents = ('accident_severity','count'),
    Average_vehicles_involved = ('vehicles_involved','mean'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')

).round({
    'Average_vehicles_involved': 2,
    'avg_casualties': 2,
    'avg_risk_score': 2
}).reset_index())

result = result.sort_values(
    by='Average_vehicles_involved',
    ascending=False
)


result

,state,total_accidents,Average_vehicles_involved,avg_casualties,avg_risk_score
6,West Bengal,2559,3.02,1.76,0.43
4,Tamil Nadu,2575,3.01,1.77,0.44
2,Maharashtra,5009,3.01,1.74,0.44
3,Punjab,2577,2.98,1.70,0.44
0,Delhi,2433,2.97,1.72,0.44
1,Karnataka,2438,2.97,1.69,0.44
5,Telangana,2409,2.97,1.68,0.44


**Q8. Highest average risk score by weekday**

In [ ]:
result = df.groupby(['state','day_of_week']).agg(
    total_accidents = ('accident_severity','count'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2}).reset_index()

result = result[['state','day_of_week','total_accidents','avg_casualties','avg_risk_score']]
result.sort_values(by=['state','avg_risk_score'],ascending=[True,False])

,state,day_of_week,total_accidents,avg_casualties,avg_risk_score
2,Delhi,Saturday,370,1.76,0.45
3,Delhi,Sunday,360,1.76,0.45
4,Delhi,Thursday,352,1.71,0.45
5,Delhi,Tuesday,337,1.78,0.44
6,Delhi,Wednesday,340,1.72,0.44
0,Delhi,Friday,314,1.65,0.42
1,Delhi,Monday,360,1.62,0.42
12,Karnataka,Tuesday,370,1.62,0.46
8,Karnataka,Monday,337,1.72,0.45
10,Karnataka,Sunday,330,1.72,0.44


**Q9. Average risk score under different weather**

In [ ]:
result = df.groupby(['state','weather']).agg(
    total_accidents = ('accident_severity','count'),
    Average_vehicles_involved = ('vehicles_involved','mean'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2,'Average_vehicles_involved': 2}).reset_index()

result = result[['state','weather','total_accidents','Average_vehicles_involved','avg_casualties','avg_risk_score']]
result.sort_values(by=['state','avg_risk_score'],ascending=[True,False])

,state,weather,total_accidents,Average_vehicles_involved,avg_casualties,avg_risk_score
1,Delhi,fog,769,2.93,1.70,0.60
2,Delhi,rain,838,2.93,1.72,0.48
0,Delhi,clear,826,3.03,1.73,0.24
4,Karnataka,fog,824,2.95,1.70,0.59
5,Karnataka,rain,796,3.05,1.71,0.49
3,Karnataka,clear,818,2.92,1.67,0.24
7,Maharashtra,fog,1640,2.97,1.72,0.59
8,Maharashtra,rain,1664,3.03,1.76,0.49
6,Maharashtra,clear,1705,3.02,1.74,0.24
10,Punjab,fog,867,2.98,1.75,0.59


# **City Analysis**

**Q1. High-risk cities**

In [ ]:
result = (df.groupby(['state','city']).agg(
    total_accident = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')).round({
    'avg_risk_score': 2
}).reset_index()
    .sort_values(by=['state','avg_risk_score'],ascending=[True,False]))
result

,state,city,total_accident,total_casualties,avg_risk_score
0,Delhi,Delhi,2433,4175,0.44
1,Karnataka,Bangalore,2438,4128,0.44
3,Maharashtra,Pune,2517,4419,0.44
2,Maharashtra,Mumbai,2492,4297,0.43
4,Punjab,Chandigarh,2577,4377,0.44
5,Tamil Nadu,Chennai,2575,4567,0.44
6,Telangana,Hyderabad,2409,4058,0.44
7,West Bengal,Kolkata,2559,4508,0.43


**Q2. Fatal accident percentage by city**

In [ ]:
df1 = df.groupby(['state','city']).agg(
    total_accidents = ('accident_severity','count'),
).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby(['state','city'])
      .agg(fatal_accidents = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on=['state','city'])
result['fatal_rate'] = (
    result['fatal_accidents'] / result['total_accidents'] * 100
).round(2)
result = result[['state','city','total_accidents','fatal_accidents','fatal_rate']]
result.sort_values(by=['state','fatal_rate'],ascending=[True,False])

,state,city,total_accidents,fatal_accidents,fatal_rate
0,Delhi,Delhi,2433,382,15.70
1,Karnataka,Bangalore,2438,384,15.75
3,Maharashtra,Pune,2517,398,15.81
2,Maharashtra,Mumbai,2492,359,14.41
4,Punjab,Chandigarh,2577,378,14.67
5,Tamil Nadu,Chennai,2575,371,14.41
6,Telangana,Hyderabad,2409,338,14.03
7,West Bengal,Kolkata,2559,377,14.73


# **Cause Analysis**

**Q1. Which accident causes are most common?**

In [ ]:
cause_accidents = (
    df.groupby('cause')
      .agg(
          total_accidents=('accident_id', 'count')
      )
      .reset_index()
      .sort_values(by='total_accidents', ascending=False)
)

cause_accidents

,cause,total_accidents
0,distraction,4026
2,overspeeding,4025
4,weather,3997
1,drunk driving,3978
3,poor road,3974


**Q2. What is the percentage distribution of fatal accidents by cause?**

In [ ]:
fatal_df = df[df['accident_severity'] == 'fatal']

cause_fatal = (
    fatal_df['cause']
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
      .reset_index()
)

cause_fatal.columns = ['cause', 'fatal_accident_percentage']

cause_fatal

,cause,fatal_accident_percentage
0,poor road,20.86
1,drunk driving,20.69
2,overspeeding,20.19
3,weather,20.12
4,distraction,18.15


**Q3. Which causes have the highest average risk score?**

In [ ]:
cause_risk = (
    df.groupby('cause')
      .agg(
          average_risk_score=('risk_score', 'mean')
      )
      .round(3)
      .reset_index()
      .sort_values(by='average_risk_score', ascending=False)
)

cause_risk

,cause,average_risk_score
2,overspeeding,0.440
4,weather,0.440
3,poor road,0.440
1,drunk driving,0.439
0,distraction,0.429


**Q4. Which accident causes contribute the highest cumulative percentage of accidents? (Pareto Analysis)**

In [ ]:
cause_pareto = (
    df.groupby('cause')
      .agg(
          total_accidents=('accident_id', 'count')
      )
      .reset_index()
      .sort_values(by='total_accidents', ascending=False)
)

cause_pareto['cumulative_percentage'] = (
    cause_pareto['total_accidents'].cumsum()
    / cause_pareto['total_accidents'].sum()
    * 100
).round(2)

cause_pareto

,cause,total_accidents,cumulative_percentage
0,distraction,4026,20.13
2,overspeeding,4025,40.26
4,weather,3997,60.24
1,drunk driving,3978,80.13
3,poor road,3974,100.00


# **Severity & Risk Analysis**

Q1. Which risk quartile has the highest average severity index?

In [ ]:
risk_category_distribution = (
    df.groupby('risk_score')
      .agg(
          total_accidents=('accident_id', 'count')
      )
      .reset_index()
      .sort_values(by='total_accidents', ascending=False)
)

risk_category_distribution

,risk_score,total_accidents
6,0.45,6421
0,0.10,3149
10,0.65,2740
2,0.25,2111
3,0.30,1664
13,0.80,1508
9,0.60,1304
14,0.85,298
5,0.40,269
17,1.00,227


# **Time Analysis**

**Q1. Peak vs Non-Peak hours**

In [ ]:
result = (
    df.groupby('is_peak_hour')
      .agg(
          total_accidents=('accident_id', 'count'),
          average_severity_index=('severity_index', 'mean'),
          average_risk_score=('risk_score', 'mean'),
          average_casualties=('casualties', 'mean')
      )
      .round(2)
      .reset_index()
)

result

,is_peak_hour,total_accidents,average_severity_index,average_risk_score,average_casualties
0,Non Peak Hour,15052,6.3,0.38,1.72
1,Peak Hour,4948,6.4,0.60,1.74


**Q2. Weekend vs Weekday**

In [ ]:
result = (
    df.groupby('is_weekend')
      .agg(
          total_accidents=('accident_id', 'count'),
          average_severity_index=('severity_index', 'mean'),
          average_risk_score=('risk_score', 'mean'),
          average_casualties=('casualties', 'mean')
      )
      .round(2)
      .reset_index()
)

result

,is_weekend,total_accidents,average_severity_index,average_risk_score,average_casualties
0,Weekday,14277,6.31,0.44,1.73
1,Weekend,5723,6.37,0.44,1.73


**Q3. Dangerous hours**

In [ ]:
df1 = df.groupby('hour').agg(
    total_accident = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby('hour')
      .agg(fatal_accident = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2,on='hour')
result['fatal_rate'] = (
    result['fatal_accident'] / result['total_accident'] * 100
).round(2)

result = result.sort_values(
    by='fatal_rate',
    ascending=False
)

result = result[['hour','total_accident','total_casualties','fatal_accident','fatal_rate']]
result

,hour,total_accident,total_casualties,fatal_accident,fatal_rate
10,10,789,1369,136,17.24
18,18,823,1454,141,17.13
12,12,866,1586,147,16.97
23,23,825,1387,140,16.97
16,16,787,1451,133,16.90
4,4,805,1367,132,16.40
3,3,824,1512,133,16.14
21,21,830,1369,133,16.02
0,0,840,1457,129,15.36
5,5,827,1377,125,15.11


**Q4. Dangerous day of week**

In [ ]:
df.groupby(['state','day_of_week']).agg(
    total_accident = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')).round({
    'avg_risk_score': 2,
}).reset_index().sort_values(by=['state','total_accident'],ascending=[True,False])

,state,day_of_week,total_accident,total_casualties,avg_risk_score
2,Delhi,Saturday,370,652,0.45
1,Delhi,Monday,360,584,0.42
3,Delhi,Sunday,360,633,0.45
4,Delhi,Thursday,352,602,0.45
6,Delhi,Wednesday,340,585,0.44
5,Delhi,Tuesday,337,600,0.44
0,Delhi,Friday,314,519,0.42
9,Karnataka,Saturday,374,616,0.43
12,Karnataka,Tuesday,370,601,0.46
13,Karnataka,Wednesday,356,626,0.43


**Q5. festival periods**

In [ ]:
df1 = df.groupby('festival').agg(
    total_accidents = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_risk_score':2}).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby('festival')
      .agg(fatal_accident = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on='festival')
result = result[['festival','total_accidents','fatal_accident','total_casualties','avg_risk_score']]
result.sort_values(by='fatal_accident',ascending=False)

,festival,total_accidents,fatal_accident,total_casualties,avg_risk_score
3,No Festival,19885,2965,34295,0.44
2,Holi,38,13,78,0.52
1,Eid,34,6,71,0.60
0,Diwali,31,3,60,0.59


**Q6. Riskiest weekday by weather**

In [ ]:
result = (
    df.groupby(['weather','day_of_week']).agg(
    total_accidents = ('accident_severity','count'),
    total_casualties = ('casualties','sum'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_risk_score':2}).reset_index()
.sort_values(by=['weather','total_accidents'],ascending=[True,False])
)

result

,weather,day_of_week,total_accidents,total_casualties,avg_risk_score
1,clear,Monday,1005,1763,0.24
3,clear,Sunday,963,1674,0.24
5,clear,Tuesday,957,1608,0.23
4,clear,Thursday,950,1608,0.24
0,clear,Friday,947,1633,0.23
2,clear,Saturday,940,1580,0.23
6,clear,Wednesday,928,1625,0.23
8,fog,Monday,999,1673,0.59
11,fog,Thursday,976,1731,0.59
10,fog,Sunday,971,1640,0.59


**Q7. High-risk hours**

In [ ]:
df1 = df.groupby(['weather','hour']).agg(
    total_accidents = ('accident_severity','count'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2}).reset_index()

df2 = (df[df['accident_severity'] == 'fatal']
      .groupby(['weather','hour'])
      .agg(fatal_accidents = ('accident_severity','count')).reset_index())

result = pd.merge(df1,df2, on=['weather','hour'])
result = result[['weather','hour','total_accidents','fatal_accidents','avg_casualties','avg_risk_score']]
result.sort_values(by=['weather','fatal_accidents'],ascending=[True,False])

,weather,hour,total_accidents,fatal_accidents,avg_casualties,avg_risk_score
10,clear,10,267,52,1.61,0.41
12,clear,12,297,46,1.87,0.20
1,clear,1,282,45,1.76,0.16
19,clear,19,296,45,1.90,0.40
2,clear,2,300,44,1.55,0.15
...,...,...,...,...,...,...
63,rain,15,272,34,1.78,0.44
62,rain,14,285,33,1.75,0.44
53,rain,5,262,31,1.56,0.46
68,rain,20,281,31,1.48,0.44


**Q8. Weekend vs Weekday traffic density**

In [ ]:
result = df.groupby(['traffic_density','is_weekend']).agg(
    total_accidents = ('accident_severity','count'),
    Average_vehicles_involved = ('vehicles_involved','mean'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2,'Average_vehicles_involved': 2}).reset_index()

result = result[['traffic_density','is_weekend','total_accidents','Average_vehicles_involved','avg_casualties','avg_risk_score']]
result.sort_values(by=['traffic_density','avg_risk_score'],ascending=[True,False])

,traffic_density,is_weekend,total_accidents,Average_vehicles_involved,avg_casualties,avg_risk_score
1,high,Weekend,1990,3.01,1.76,0.60
0,high,Weekday,5044,2.99,1.72,0.59
2,low,Weekday,5096,2.96,1.72,0.34
3,low,Weekend,1971,3.00,1.73,0.34
4,medium,Weekday,4137,3.01,1.74,0.37
5,medium,Weekend,1762,3.02,1.70,0.37


**Q9. Causes during peak & non-peak hours**

In [ ]:
result = df.groupby(['cause','is_peak_hour']).agg(
    total_accidents = ('accident_severity','count'),
    Average_vehicles_involved = ('vehicles_involved','mean'),
    avg_casualties = ('casualties','mean'),
    avg_risk_score = ('risk_score','mean')
).round({'avg_casualties':2,'avg_risk_score':2,'Average_vehicles_involved':2}).reset_index()

result = result[['cause','is_peak_hour','total_accidents','Average_vehicles_involved','avg_casualties','avg_risk_score']]
result.sort_values(by=['cause','avg_risk_score'],ascending=[True,False])

,cause,is_peak_hour,total_accidents,Average_vehicles_involved,avg_casualties,avg_risk_score
1,distraction,Peak Hour,967,2.97,1.69,0.59
0,distraction,Non Peak Hour,3059,3.01,1.71,0.38
3,drunk driving,Peak Hour,972,2.98,1.73,0.61
2,drunk driving,Non Peak Hour,3006,2.99,1.73,0.38
5,overspeeding,Peak Hour,1032,2.97,1.68,0.61
4,overspeeding,Non Peak Hour,2993,3.02,1.74,0.38
7,poor road,Peak Hour,997,3.09,1.78,0.60
6,poor road,Non Peak Hour,2977,2.97,1.72,0.39
9,weather,Peak Hour,980,2.97,1.81,0.61
8,weather,Non Peak Hour,3017,2.96,1.72,0.38


**Q10.** **highest number of accidents(week)**

In [ ]:
df.groupby('day_of_week')['accident_id'].count().sort_values(ascending=False)

,accident_id
day_of_week,
Monday,2966
Friday,2879
Tuesday,2871
Saturday,2867
Sunday,2856
Thursday,2835
Wednesday,2726


# **Weather & Environment Analysis**

**Q1. Weather vs severity**

In [ ]:
pd.pivot_table(
    data = df,
    index= 'weather',
    columns= 'accident_severity',
    values= 'accident_id',
    fill_value= 0,
    aggfunc= 'count'

).reset_index()

accident_severity,weather,fatal,major,minor
0,clear,992,2017,3681
1,fog,995,1975,3663
2,rain,1000,1996,3681


**Q2. Which weather condition records the highest number of road accidents?**

In [ ]:
weather_accidents = (
    df.groupby('weather')
      .agg(
          total_accidents=('accident_id', 'count')
      )
      .reset_index()
      .sort_values(by='total_accidents', ascending=False)
)

weather_accidents

,weather,total_accidents
0,clear,6690
2,rain,6677
1,fog,6633


**Q3. What is the percentage distribution of fatal accidents across different weather conditions?**

In [ ]:
fatal_df = df[df['accident_severity'] == 'fatal']

weather_fatal = (
    fatal_df['weather']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

weather_fatal.columns = ['weather', 'fatal_accident_percentage']

weather_fatal

,weather,fatal_accident_percentage
0,rain,33.48
1,fog,33.31
2,clear,33.21


**Q4. Which weather condition has the highest average risk score?**

In [ ]:
weather_risk = (
    df.groupby('weather')
      .agg(
          total_accidents=('accident_id', 'count'),
          average_risk_score=('risk_score', 'mean')
      )
      .round({'average_risk_score': 3})
      .reset_index()
      .sort_values(by='average_risk_score', ascending=False)
)

weather_risk

,weather,total_accidents,average_risk_score
1,fog,6633,0.589
2,rain,6677,0.488
0,clear,6690,0.237


**Q5. How does traffic density vary under different weather conditions?**

In [ ]:
weather_traffic = (
    pd.crosstab(
        df['weather'],
        df['traffic_density'],
        normalize='index'
    )
    .mul(100)
    .round(2)
    .reset_index()
)

weather_traffic

traffic_density,weather,high,low,medium
0,clear,34.99,35.72,29.28
1,fog,35.10,35.28,29.62
2,rain,35.42,35.00,29.58


**Q6. How does visibility vary under different weather conditions?**

In [ ]:
weather_visibility = (
    pd.crosstab(
        df['weather'],
        df['visibility'],
        normalize='index'
    )
    .mul(100)
    .round(2)
    .reset_index()
)

weather_visibility

visibility,weather,high,low,medium
0,clear,100.0,0.00,0.00
1,fog,0.0,100.00,0.00
2,rain,0.0,50.23,49.77


**Q7. What is the average temperature under different weather conditions?**

In [ ]:
weather_temperature = (
    df.groupby('weather')
      .agg(
          average_temperature=('temperature', 'mean')
      )
      .round({'average_temperature': 2})
      .reset_index()
      .sort_values(by='average_temperature', ascending=False)
)

weather_temperature

,weather,average_temperature
1,fog,27.69
0,clear,27.58
2,rain,27.47


# **Road Infrastructure Analysis**

**Q1. Which road type records the highest number of road accidents?**

In [ ]:
road_accidents = (
    df.groupby('road_type')
      .size()
      .reset_index(name='total_accidents')
      .sort_values(by='total_accidents', ascending=False)
)

road_accidents

,road_type,total_accidents
2,urban,6745
1,rural,6639
0,highway,6616


**Q2. What is the distribution of fatal accidents across different road types?**

In [ ]:
fatal_df = df[df['accident_severity'] == 'fatal']

fatal_df = (fatal_df['road_type'].value_counts(normalize=True)*100).round(2).reset_index()
fatal_df.columns = ['road_type','percentage of fatal accident']
fatal_df

,road_type,percentage of fatal accident
0,urban,34.78
1,highway,32.91
2,rural,32.31


**Q3. Which road type has the highest average risk score?**

In [ ]:
road_accidents = (
    df.groupby('road_type').agg(
        average_risk_score = ('risk_score','mean')
    ).round({'average_risk_score':3})
      .reset_index()
      .sort_values(by='average_risk_score', ascending=False)
)

road_accidents

,road_type,average_risk_score
2,urban,0.441
0,highway,0.437
1,rural,0.435


**Q4. How does accident severity vary across different road types?**

In [ ]:
road_accidents = (
    df.groupby('road_type').agg(
        total_accidents = ('accident_id','count'),
        average_severity_index = ('severity_index','mean')
    ).round({'average_severity_index':2})
      .reset_index()
      .sort_values(by='average_severity_index', ascending=False)
)

road_accidents

,road_type,total_accidents,average_severity_index
2,urban,6745,6.37
0,highway,6616,6.31
1,rural,6639,6.30


**Q5. Does accident frequency increase or decrease as the number of lanes changes?**

In [ ]:
road_accidents = (
    df.groupby('lanes').agg(
        total_accidents = ('accident_id','count'),
    )
      .reset_index()
      .sort_values(by='total_accidents', ascending=False)
)

road_accidents

,lanes,total_accidents
3,4,3383
0,1,3382
4,5,3363
2,3,3334
1,2,3270
5,6,3268


**Q6. Does the presence of traffic signals influence accident frequency?**

In [ ]:
road_accidents = (
    df.groupby('traffic_signal').agg(
        total_accidents = ('accident_id','count'),
    )
      .reset_index().replace({'traffic_signal': {0: 'No Signal', 1: 'Traffic Signal'}})
      .sort_values(by='total_accidents', ascending=False)
)

road_accidents

,traffic_signal,total_accidents
0,No Signal,10003
1,Traffic Signal,9997


**Q7. How does accident severity differ between roads with and without traffic signals?**

In [ ]:
df.groupby('traffic_signal').agg(
    total_accidents = ('accident_id','count'),
     average_severity_index = ('severity_index','mean')
     ).round({'average_severity_index':2}).reset_index().sort_values(by='average_severity_index', ascending=False)

,traffic_signal,total_accidents,average_severity_index
1,1,9997,6.41
0,0,10003,6.24


# 📊 Overall KPI & Business Analysis Summary

## 🚦 Project Overview

This KPI and Business Analysis explored road accident patterns across India by examining geographical distribution, accident severity, temporal trends, environmental conditions, road infrastructure, and accident causes. Rather than focusing only on accident counts, the analysis aimed to identify the factors associated with higher accident risk, greater accident severity, and increased fatalities. The findings provide meaningful insights that can support evidence-based decision-making and help improve road safety planning.

---

# 📈 Executive KPI Highlights

The analysis was performed on **20,000** road accident records collected from multiple cities and states across India.

### Key Performance Indicators (KPIs)

- 🚗 **Total Road Accidents:** **20,000**
- 💀 **Total Fatal Accidents:** **2,987**
- 📉 **Overall Fatality Rate:** **14.94%**
- ⚠️ **Average Risk Score:** **0.44**
- 🚨 **Average Severity Index:** **6.32**
- 🩹 **Average Casualties per Accident:** **1.72**
- 🚙 **Average Vehicles Involved per Accident:** **3.00**

These KPIs provide a high-level overview of the dataset and establish the baseline for evaluating accident patterns throughout the analysis.

---

# 🗺️ Geographic Analysis

The geographical analysis revealed that accident patterns differ significantly across states and cities. While certain regions experienced a higher number of accidents, others recorded higher accident severity or fatality rates. This indicates that accident frequency alone does not fully represent road safety conditions.

### 📍 State-Level Highlights

- Maharashtra recorded the highest number of road accidents.
- Karnataka reported the highest fatal accident rate.
- Punjab had the highest average risk score.
- Tamil Nadu recorded the highest average casualties per accident.

These findings suggest that each state faces unique road safety challenges and requires region-specific policies rather than a one-size-fits-all approach.

### 🏙️ City-Level Highlights

- Chandigarh recorded the highest number of accidents.
- Pune reported the highest number of fatal accidents.
- Chandigarh also recorded the highest average risk score.

The city-level analysis highlights urban areas where traffic management, infrastructure upgrades, and road safety initiatives should be prioritized.

---

# ⏰ Time-Based Analysis

Accident frequency and severity varied across different time periods, indicating that traffic conditions and travel behavior have a significant impact on road safety.

### Key Findings

- 🚦 Peak accident periods occurred between **9:00–10:00 AM** and **12:00–1:00 PM**.
- 📅 Weekday accidents were more frequent than weekend accidents.
- 🎉 Festival periods recorded a higher average risk score.
- 📌 Monday experienced the highest number of reported accidents.

These trends can help traffic authorities optimize enforcement schedules, improve emergency response planning, and deploy additional resources during high-risk periods.

---

# 🌦️ Weather & Environmental Analysis

Environmental conditions played an important role in influencing accident frequency and overall road risk.

### Key Findings

- ☀️ Clear weather recorded the highest number of accidents, likely because it represents the most common driving condition.
- 🌫️ Poor visibility was associated with an increased likelihood of accidents.
- 🚗 Heavy traffic density contributed to higher accident frequency.
- 🌡️ Temperature showed only a weak relationship with accident occurrence compared to other environmental factors.

These findings highlight the importance of weather-aware traffic management systems and public safety advisories during adverse driving conditions.

---

# 🛣️ Road Infrastructure Analysis

Road infrastructure characteristics significantly influenced both accident frequency and accident severity.

### Key Findings

- 🏙️ Urban roads experienced the highest number of accidents.
- 🚨 Urban roads also recorded the highest average accident severity.
- 🛣️ Four-lane roads reported the highest accident concentration.
- 🚦 Locations without traffic signals experienced higher accident severity.

These results suggest that improving road infrastructure, optimizing traffic control systems, and strengthening intersection management could contribute to reducing accident severity.

---

# 🚘 Cause Analysis

Understanding the primary causes of accidents helps identify where preventive measures can have the greatest impact.

### Key Findings

- 📱 Driver distraction was the leading cause of road accidents.
- 🛣️ Poor road conditions contributed to the highest proportion of fatal accidents.
- 📊 Pareto analysis showed that approximately **80%** of accidents were associated with the **top four accident causes**.

This finding indicates that addressing a small number of major accident causes could significantly reduce overall accident frequency and improve road safety outcomes.

---

# ⚠️ Severity & Risk Analysis

Severity and risk analysis provided a deeper understanding of accident impact beyond simple accident counts.

### Key Findings

- 📈 Risk Quartile **Q2** contained the largest number of accidents.
- 🔴 Risk Quartile **Q4** represented the highest-risk and most severe accident group.
- 🚑 High-severity accidents generally involved more casualties and a greater number of vehicles.
- 🔗 Correlation analysis suggested that accident severity is influenced by multiple interacting factors rather than a single dominant variable.

These findings demonstrate the value of risk segmentation when prioritizing road safety initiatives and allocating emergency response resources.

---

# 💡 Overall Business Insights

The analysis demonstrates that road accidents are influenced by a combination of geographical, temporal, environmental, infrastructural, and behavioral factors rather than a single cause.

Some of the most important insights include:

- 📍 Accident frequency and accident severity do not always occur in the same locations.
- ⚠️ Regions with more accidents are not necessarily the regions with the highest accident risk.
- 🌦️ Environmental conditions such as poor visibility and heavy traffic increase accident probability.
- 🛣️ Road infrastructure has a measurable impact on accident severity.
- 📊 A small number of accident causes contribute to the majority of accident cases, supporting the Pareto Principle.
- ⏰ Time-based accident patterns create opportunities for targeted traffic enforcement.
- 🚑 Risk-based analysis enables authorities to prioritize limited road safety resources more effectively.

---

# 🎯 Business Recommendations

Based on the complete analysis, the following recommendations are proposed:

1. 🚦 Prioritize road safety investments in high-risk states and cities.
2. 👮 Increase traffic enforcement during peak hours, weekdays, and festival periods.
3. 🛣️ Improve road infrastructure in accident-prone locations.
4. 🚥 Install or optimize traffic signals at high-risk intersections.
5. 🌧️ Implement weather-based traffic advisory systems during adverse conditions.
6. 📢 Strengthen awareness campaigns focusing on driver distraction and poor road conditions.
7. 🚑 Improve emergency response capabilities in regions with higher accident severity.
8. 📊 Continuously monitor road safety KPIs through interactive dashboards.
9. 📈 Prioritize interventions based on accident risk and severity instead of accident count alone.
10. 🔄 Regularly update accident analysis using newly collected data to evaluate the effectiveness of implemented road safety measures.

---

# ✅ Conclusion

This KPI and Business Analysis successfully transformed raw road accident records into meaningful business insights through a structured analytical approach. By combining executive KPIs, geographical analysis, temporal trends, environmental assessment, road infrastructure evaluation, cause analysis, and severity analysis, the project provides a comprehensive understanding of road accident patterns across different regions and conditions.

The insights generated from this analysis can support transportation authorities, policymakers, and urban planners in developing targeted road safety strategies, improving infrastructure planning, optimizing traffic management, and reducing accident severity. Furthermore, this analysis establishes a strong foundation for the interactive dashboard and future predictive modeling initiatives.